#### <b>Import Statements & Data Download</b>
The CSV files referenced in this notebook were downloaded from the following public repository that contains scraped transfer data for top soccer leagues (English Premier League, La Liga, Ligue 1, Serie A, Bundesliga, Eredivisie, Süper Lig, and Saudi Pro League) from 1992 to 2025: https://github.com/eordo/transfermarkt-data/blob/master/README.md

This data was scraped from the following website: https://www.transfermarkt.us/premier-league/startseite/wettbewerb/GB1

For this notebook, download the CSV files from 2015 to 2025 for the English Premier League.

In [7]:
import numpy as np
import pandas as pd

# Recommended: consolidate the transfer data CSV files into a sub-folder in your working repository
data_folder = "Transfer_Data" # rename based on folder name

#### <b>Data Consolidation</b>
Created the following pandas dataframe that captures descriptive information and transfer activity for English Premier League clubs from 2015 to 2026. The dataframe is titled `epl_transfer_data`.

Below are the columns included in `epl_transfer_data` along with their definitions.

1. season — season start year  
2. league — league in which the club competes  
3. club — Premier League club associated with the transfer  
4. window — transfer window in which the transaction occurred, either summer or winter  
5. movement — direction of the transfer relative to the Premier League club, either in or out  
6. player_name — name of the player involved in the transfer  
7. player_id — Transfermarkt player identifier  
8. age — player age at the time of the transfer  
9. nationality — player nationality  
10. position — full description of the player’s primary position  
11. pos — abbreviated player position  
12. market_value — estimated market value of the player at the time of the transfer  
13. dealing_club — club on the opposite side of the transaction  
14. dealing_country — country of the club on the opposite side of the transaction  
15. fee — reported transfer or loan fee associated with the transaction; a value of 0 generally represents no reported fee, while missing values indicate that a fee is unavailable or not reported  
16. is_loan — indicator for whether the transaction was a loan, where 1 indicates a loan and 0 indicates a non-loan transfer
17. Fee_Missing — indicator for whether the reported transfer or loan fee is missing, where 1 indicates a missing fee and 0 indicates that a fee value is available  
18. Market_Value_Missing — indicator for whether the player’s market value is missing, where 1 indicates a missing market value and 0 indicates that a market value is available

In [ ]:
transfer_data_list = []
for year in range(2015, 2026):
    transfer_data = pd.read_csv(f"{data_folder}/{year}.csv")
    transfer_data_list.append(transfer_data)
epl_transfer_data = pd.concat(transfer_data_list, ignore_index=True)
epl_transfer_data

,season,league,club,window,movement,player_name,player_id,age,nationality,position,pos,market_value,dealing_club,dealing_country,fee,is_loan
0,2015,Premier League,AFC Bournemouth,summer,in,Tyrone Mings,253677,22,England,Centre-Back,CB,6000000.0,Ipswich Town,England,11300000.0,0
1,2015,Premier League,AFC Bournemouth,summer,in,Max Gradel,28140,27,Cote d'Ivoire,Left Winger,LW,6500000.0,AS Saint-Étienne,France,10000000.0,0
2,2015,Premier League,AFC Bournemouth,summer,in,Glenn Murray,45693,31,England,Centre-Forward,CF,2500000.0,Crystal Palace,England,5400000.0,0
3,2015,Premier League,AFC Bournemouth,summer,in,Lee Tomlin,104586,26,England,Second Striker,SS,1750000.0,Middlesbrough FC,England,4300000.0,0
4,2015,Premier League,AFC Bournemouth,summer,in,Sylvain Distin,3356,37,France,Centre-Back,CB,750000.0,Everton FC,England,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6085,2025,Premier League,Wolverhampton Wanderers,summer,out,Craig Dawson,121477,35,England,Centre-Back,CB,1500000.0,Without Club,NaN,NaN,0
6086,2025,Premier League,Wolverhampton Wanderers,summer,out,Nasser Djiga,862297,22,Burkina Faso,Centre-Back,CB,10000000.0,Rangers FC,Scotland,0.0,1
6087,2025,Premier League,Wolverhampton Wanderers,summer,out,Pedro Lima,1193867,19,Brazil,Right-Back,RB,5000000.0,FC Porto,Portugal,0.0,1
6088,2025,Premier League,Wolverhampton Wanderers,summer,out,Carlos Forbs,742451,21,Portugal,Left Winger,LW,8000000.0,Ajax Amsterdam,Netherlands,0.0,1


#### <b>Preliminary Data Cleaning</b>

In [3]:
epl_transfer_data[epl_transfer_data.isna().any(axis = 1)]
nan_cols = []
for col in epl_transfer_data.columns:
    nan_count = len(epl_transfer_data[col][epl_transfer_data[col].isna()])
    if nan_count > 0:
        nan_cols.append(col)
market_value_cols = epl_transfer_data[epl_transfer_data["market_value"].isna()] # no unique identifiers for market_value when comparing to other columns
dealing_country_cols = epl_transfer_data[epl_transfer_data["dealing_country"].isna()] # values for the dealing_club column when dealing_country is NaN include: 'Without Club', 'Retired', 'Disqualification', 'Career break'
fee_cols = epl_transfer_data[epl_transfer_data["fee"].isna()] # values for the is_loan column when fee is NaN are always zero

In [4]:
epl_transfer_data["Fee_Missing"] = epl_transfer_data["fee"].isna().astype(int)
epl_transfer_data["Market_Value_Missing"] = epl_transfer_data["market_value"].isna().astype(int)
epl_transfer_data["dealing_country"] = epl_transfer_data["dealing_country"].fillna("None")

# epl_transfer_data["Fee_Missing"][epl_transfer_data["Fee_Missing"] == 1] # length should equal 412 rows
# epl_transfer_data["Market_Value_Missing"][epl_transfer_data["Market_Value_Missing"] == 1] # length should equal 249 rows
# epl_transfer_data["dealing_country"][epl_transfer_data["dealing_country"] == "None"] # length should equal 295 rows

#### <b>Data Export</b>
Run the following line to save the premier league transfer data to an Excel file on your device.

In [ ]:
with pd.ExcelWriter("epl_transfer_data.xlsx") as writer:
    epl_transfer_data.to_excel(writer, index = False) 